In [ ]:
using LinearAlgebra # Necesario para resolver el sistema lineal

"""
    pade_approximation(coefs_maclaurin, n, m)

Calcula los coeficientes de los polinomios P(x) y Q(x) para la aproximación de Padé.

Argumentos:
- `coefs_maclaurin`: Un vector con los coeficientes de la serie de Maclaurin (Taylor en 0) de f(x). 
                    Debe tener al menos tamaño N + 1, donde N = n + m. 
                    [a_0, a_1, ..., a_N]
- `n`: Grado del polinomio numerador P(x).
- `m`: Grado del polinomio denominador Q(x).

Retorna:
- `p`: Vector de coeficientes del numerador [p_0, ..., p_n]
- `q`: Vector de coeficientes del denominador [q_0, ..., q_m] (donde q_0 = 1)
"""
function pade_aproximacion(coefs_maclaurin::Vector{T}, n::Int, m::Int) where T <: Number
    N = n + m
    
    # Verificación
    if length(coefs_maclaurin) < N + 1
        error("Necesitas al menos $(N+1) coeficientes de Maclaurin.")
    end

    #Funcion para los coeficientes 'a'
    function a(i)
        if (i < 0 || i > length(coefs_maclaurin)-1)
            return 0.0
        else
            return coefs_maclaurin[i+1]
        end
    end

    #Encontrar los coeficientes q
    if m > 0
        # Construimos la matriz A y el vector b para el sistema Aq = b
        A = zeros(T, m, m)
        b = zeros(T, m)
        
        for i in 1:m
            for j in 1:m
                A[i, j] = a(n + i - j) #Coeficiente q_j es a_{n + i - j}
            end
            b[i] = -a(n + i) #Lado derecho -a_{n+i} 
        end
        
        #q_coeffs = [q_1, q_2, ..., q_m]
        try
            q_solve = A \ b
            q = [1.0; q_solve] #q_0 = 1 al inicio
        catch
            error("El sistema es singular.")
        end
    else
        # Si m=0 entonces es la serie de Taylor
        q = [1.0]
    end

    #Coeficientes p
    p = zeros(T, n + 1)
    
    for k in 0:n # p_k = sum(a_{k-j} * q_j) desde j=0 hasta k
        sum = 0.0
        for j in 0:k
            if j <= m # Sumamos si q_j existe
                sum += a(k - j) * q[j+1]
            end
        end
        p[k+1] = sum
    end

    return p, q
end

pade_aproximacion

In [9]:
function factorial(k)
    return prod(1:k)
end

N = 2 + 2 # n + m
a_coeffs = [1.0 / factorial(i) for i in 0:N] 

#Funcion
p, q = pade_aproximacion(a_coeffs, 2, 2)

println("f(x) = e^x  n=2, m=2:")
println("Numerador P(x): $p")
println("Denominador Q(x): $q")

# 3. Verificamos construyendo la función racional
pad_func(x) = (p[1] + p[2]*x + p[3]*x^2) / (q[1] + q[2]*x + q[3]*x^2)

println("\nx = 1.0:")
println("e^1 real:      $(exp(1.0))")
println("Aproximación:  $(pad_func(1.0))")

f(x) = e^x  n=2, m=2:
Numerador P(x): [1.0, 0.5000000000000001, 0.08333333333333341]
Denominador Q(x): [1.0, -0.4999999999999999, 0.0833333333333333]

x = 1.0:
e^1 real:      2.718281828459045
Aproximación:  2.7142857142857144
